# 11 — GPU Utilization Profiling (H100)

**Goal:** Measure GPU utilization during video inference and produce a per-second report.

Metrics collected per second:
- GPU utilization %
- GPU memory usage (MB)
- Inference FPS
- Frame latency (ms)

Uses NVML (via pynvml) for accurate GPU monitoring, synchronized with inference.

## 1 — Setup

In [ ]:
!pip install -q ultralytics pynvml opencv-python

In [ ]:
import os, sys, time, json, csv, threading
import cv2
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
VIDEO_DIR = f'{ECOCAR_ROOT}/video'
WEIGHTS_DIR = f'{ECOCAR_ROOT}/weights'
OUTPUT_DIR = '/content/gpu_profile'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PROJECT_DIR = os.path.join(ECOCAR_ROOT, 'project')
assert os.path.isdir(PROJECT_DIR), f'Project not found at {PROJECT_DIR}'
sys.path.insert(0, PROJECT_DIR)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')

## 2 — NVML GPU Monitor

In [ ]:
import pynvml

pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
gpu_info = pynvml.nvmlDeviceGetName(handle)
if isinstance(gpu_info, bytes):
    gpu_info = gpu_info.decode()
print(f'NVML device: {gpu_info}')

class GPUMonitor:
    """Background thread that polls GPU utilization every interval_ms."""

    def __init__(self, interval_ms=100):
        self.interval = interval_ms / 1000.0
        self.samples = []
        self._running = False
        self._thread = None

    def start(self):
        self._running = True
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False
        if self._thread:
            self._thread.join(timeout=2.0)

    def _poll(self):
        while self._running:
            try:
                util = pynvml.nvmlDeviceGetUtilizationRates(handle)
                mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
                self.samples.append({
                    'timestamp': time.time(),
                    'gpu_util': util.gpu,
                    'mem_util': util.memory,
                    'mem_used_mb': mem.used / (1024**2),
                    'mem_total_mb': mem.total / (1024**2),
                })
            except Exception:
                pass
            time.sleep(self.interval)

    def get_per_second_report(self, start_time):
        """Aggregate samples into per-second buckets."""
        if not self.samples:
            return []

        report = []
        current_second = 0
        bucket = []

        for s in self.samples:
            sec = int(s['timestamp'] - start_time)
            if sec < 0:
                continue
            if sec != current_second:
                if bucket:
                    report.append({
                        'second': current_second,
                        'gpu_util_avg': np.mean([b['gpu_util'] for b in bucket]),
                        'gpu_util_max': max(b['gpu_util'] for b in bucket),
                        'mem_used_avg_mb': np.mean([b['mem_used_mb'] for b in bucket]),
                        'mem_used_max_mb': max(b['mem_used_mb'] for b in bucket),
                        'n_samples': len(bucket),
                    })
                current_second = sec
                bucket = []
            bucket.append(s)

        if bucket:
            report.append({
                'second': current_second,
                'gpu_util_avg': np.mean([b['gpu_util'] for b in bucket]),
                'gpu_util_max': max(b['gpu_util'] for b in bucket),
                'mem_used_avg_mb': np.mean([b['mem_used_mb'] for b in bucket]),
                'mem_used_max_mb': max(b['mem_used_mb'] for b in bucket),
                'n_samples': len(bucket),
            })

        return report

print('GPUMonitor ready.')

## 3 — Load Model & Video

In [ ]:
from src.multitask_model import build_multitask_model
from src.utils.class_map import VEHICLE_CLASSES, NUM_VEHICLE_CLASSES

CKPT_PATH = os.path.join(WEIGHTS_DIR, 'vehicle_lane_v4_best.pt')
assert os.path.isfile(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'

ckpt = torch.load(CKPT_PATH, map_location='cuda', weights_only=False)
arch = ckpt.get('arch_config', {})

model = build_multitask_model(
    weights='yolo26s.pt',
    lane_head_type=arch.get('lane_head_type', 'transformer'),
    lane_embed_dim=arch.get('lane_embed_dim', 128),
    lane_num_heads=arch.get('lane_num_heads', 4),
    lane_depth=arch.get('lane_depth', 2),
    mask_height=arch.get('mask_height', 160),
    mask_width=arch.get('mask_width', 160),
)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
model = model.to('cuda').eval()

# Find video
video_files = sorted([
    os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR)
    if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
]) if os.path.isdir(VIDEO_DIR) else []
assert video_files, f'No videos in {VIDEO_DIR}'
INPUT_VIDEO = video_files[0]

cap = cv2.VideoCapture(INPUT_VIDEO)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_video = cap.get(cv2.CAP_PROP_FPS)
print(f'Video: {os.path.basename(INPUT_VIDEO)} ({total_frames} frames @ {fps_video:.0f} FPS)')

## 4 — Profiled Inference

In [ ]:
try:
    from ultralytics.utils.nms import non_max_suppression
except ImportError:
    from ultralytics.utils.ops import non_max_suppression

IMG_SIZE = 640
CONF_THRESH = 0.3

# Warmup
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device='cuda')
for _ in range(10):
    with torch.no_grad(), torch.amp.autocast('cuda'):
        _ = model(dummy)
torch.cuda.synchronize()
print('Warmup complete.')

# Start GPU monitor
monitor = GPUMonitor(interval_ms=100)
frame_times = []

torch.cuda.synchronize()
monitor.start()
t_start = time.time()
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t0 = time.time()

    img = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(img.astype(np.float32) / 255.0)
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to('cuda')

    with torch.no_grad(), torch.amp.autocast('cuda'):
        out = model(tensor)

    det_out = out['det_output']
    if isinstance(det_out, (tuple, list)) and len(det_out) == 2:
        y_post = det_out[0]
        if not (isinstance(y_post, torch.Tensor) and y_post.dim() == 3 and y_post.shape[-1] == 6):
            _ = non_max_suppression(y_post, conf_thres=CONF_THRESH, iou_thres=0.45, max_det=100)
    elif isinstance(det_out, torch.Tensor):
        _ = non_max_suppression(det_out, conf_thres=CONF_THRESH, iou_thres=0.45, max_det=100)
    _ = torch.sigmoid(out['lane_logits'])

    torch.cuda.synchronize()
    t1 = time.time()

    frame_times.append((t1 - t0) * 1000)
    frame_idx += 1

    if frame_idx % 200 == 0:
        print(f'  Frame {frame_idx}/{total_frames}')

torch.cuda.synchronize()
t_end = time.time()

monitor.stop()
cap.release()

total_time = t_end - t_start
avg_fps = frame_idx / total_time
avg_latency = np.mean(frame_times)
p95_latency = np.percentile(frame_times, 95)
p99_latency = np.percentile(frame_times, 99)

print(f'\nProfiling complete:')
print(f'  Frames:       {frame_idx}')
print(f'  Total time:   {total_time:.1f}s')
print(f'  Avg FPS:      {avg_fps:.1f}')
print(f'  Avg latency:  {avg_latency:.2f}ms')
print(f'  P95 latency:  {p95_latency:.2f}ms')
print(f'  P99 latency:  {p99_latency:.2f}ms')

## 5 — Per-Second GPU Utilization Report

In [ ]:
report = monitor.get_per_second_report(t_start)

print(f'Per-second GPU utilization report ({len(report)} seconds):')
print(f'{"Second":>6} {"GPU%":>6} {"GPU Max%":>9} {"Mem (MB)":>10} {"Samples":>8}')
print('-' * 45)
for r in report:
    print(f'{r["second"]:>6} {r["gpu_util_avg"]:>5.1f}% {r["gpu_util_max"]:>8.1f}% '
          f'{r["mem_used_avg_mb"]:>9.0f} {r["n_samples"]:>8}')

# Save report CSV
report_csv = f'{OUTPUT_DIR}/gpu_utilization_per_second.csv'
with open(report_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=[
        'second', 'gpu_util_avg', 'gpu_util_max',
        'mem_used_avg_mb', 'mem_used_max_mb', 'n_samples'
    ])
    w.writeheader()
    w.writerows(report)
print(f'\nReport saved: {report_csv}')

# Save frame latencies
latency_csv = f'{OUTPUT_DIR}/frame_latencies.csv'
with open(latency_csv, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['frame', 'latency_ms'])
    for i, t in enumerate(frame_times):
        w.writerow([i, f'{t:.3f}'])
print(f'Frame latencies saved: {latency_csv}')

## 6 — Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# GPU utilization over time
seconds = [r['second'] for r in report]
gpu_avg = [r['gpu_util_avg'] for r in report]
gpu_max = [r['gpu_util_max'] for r in report]

axes[0, 0].fill_between(seconds, gpu_avg, alpha=0.3, color='blue')
axes[0, 0].plot(seconds, gpu_avg, label='Avg', color='blue')
axes[0, 0].plot(seconds, gpu_max, label='Max', color='red', alpha=0.5)
axes[0, 0].set_title('GPU Utilization per Second')
axes[0, 0].set_ylabel('GPU Utilization (%)')
axes[0, 0].set_xlabel('Time (seconds)')
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 105)

# Memory usage
mem_avg = [r['mem_used_avg_mb'] / 1024 for r in report]
axes[0, 1].plot(seconds, mem_avg, color='green')
axes[0, 1].set_title('GPU Memory Usage per Second')
axes[0, 1].set_ylabel('Memory (GB)')
axes[0, 1].set_xlabel('Time (seconds)')

# Frame latency distribution
axes[1, 0].hist(frame_times, bins=50, color='orange', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(avg_latency, color='red', linestyle='--', label=f'Avg: {avg_latency:.1f}ms')
axes[1, 0].axvline(p95_latency, color='purple', linestyle='--', label=f'P95: {p95_latency:.1f}ms')
axes[1, 0].set_title('Frame Latency Distribution')
axes[1, 0].set_xlabel('Latency (ms)')
axes[1, 0].legend()

# FPS over time (rolling window)
window = 30
if len(frame_times) > window:
    rolling_fps = [1000.0 / np.mean(frame_times[max(0,i-window):i+1]) for i in range(len(frame_times))]
    axes[1, 1].plot(rolling_fps, linewidth=0.5, color='teal')
    axes[1, 1].axhline(avg_fps, color='red', linestyle='--', label=f'Avg: {avg_fps:.0f} FPS')
    axes[1, 1].set_title(f'Rolling FPS (window={window})')
    axes[1, 1].set_xlabel('Frame')
    axes[1, 1].set_ylabel('FPS')
    axes[1, 1].legend()

plt.suptitle(f'GPU Profiling Report — {gpu_name}', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/gpu_profile.png', dpi=150)
plt.show()

## 7 — Summary

In [ ]:
# Print final summary
avg_gpu = np.mean(gpu_avg) if gpu_avg else 0
max_gpu = max(gpu_max) if gpu_max else 0
avg_mem = np.mean(mem_avg) if mem_avg else 0

print('=' * 55)
print(f'  GPU PROFILING SUMMARY')
print('=' * 55)
print(f'  GPU:              {gpu_name}')
print(f'  Video:            {os.path.basename(INPUT_VIDEO)}')
print(f'  Frames:           {frame_idx}')
print(f'  Duration:         {total_time:.1f}s')
print(f'  Avg FPS:          {avg_fps:.1f}')
print(f'  Avg latency:      {avg_latency:.2f}ms')
print(f'  P95 latency:      {p95_latency:.2f}ms')
print(f'  P99 latency:      {p99_latency:.2f}ms')
print(f'  Avg GPU util:     {avg_gpu:.1f}%')
print(f'  Peak GPU util:    {max_gpu:.1f}%')
print(f'  Avg GPU memory:   {avg_mem:.2f} GB')
print('=' * 55)

# Copy results to Drive
import shutil
drive_out = f'{ECOCAR_ROOT}/profiling'
os.makedirs(drive_out, exist_ok=True)
for f in ['gpu_utilization_per_second.csv', 'frame_latencies.csv', 'gpu_profile.png']:
    src = f'{OUTPUT_DIR}/{f}'
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
print(f'\nResults copied to {drive_out}')